# Implementazione EME — prime verifiche

Parto dal codice della repository ufficiale e provo a ricostruire gradualmente l'architettura su YOLOv5-Small. In questa fase lavoro con input sintetici: l'obiettivo è controllare che i moduli siano collegati correttamente, non valutare ancora le predizioni.

**File da conservare a parte:** `models/yolov5s_eme.yaml`, `models/eme_weak_sup.py` e la versione modificata di `models/common.py`. Sono file creati o modificati nell'ambiente Colab e non vengono inclusi automaticamente nel notebook.

In [ ]:
!git clone --depth 1 https://github.com/XueZ-phd/Efficient-RGB-T-Early-Fusion-Detection.git

Cloning into 'Efficient-RGB-T-Early-Fusion-Detection'...
remote: Enumerating objects: 2249, done.
remote: Counting objects: 100% (2249/2249), done.
remote: Compressing objects: 100% (1711/1711), done.
remote: Total 2249 (delta 490), reused 2237 (delta 488), pack-reused 0 (from 0)
Receiving objects: 100% (2249/2249), 164.39 MiB | 23.22 MiB/s, done.
Resolving deltas: 100% (490/490), done.
Updating files: 100% (2019/2019), done.


### Controllo della repository

Dopo il clone mi sposto nella cartella `yolov5` e verifico che ci siano i file principali da cui partire.

In [ ]:
%cd /content/Efficient-RGB-T-Early-Fusion-Detection/yolov5

from pathlib import Path

files = [
    "train_rgbt_earlyFusion.py",
    "train_rgbt_distillation_earlyFusion.py",
    "models/yolov5s.yaml",
    "requirements.txt"
]

for file in files:
    print(file, "✅" if Path(file).exists() else "❌")

/content/Efficient-RGB-T-Early-Fusion-Detection/yolov5
train_rgbt_earlyFusion.py ✅
train_rgbt_distillation_earlyFusion.py ✅
models/yolov5s.yaml ✅
requirements.txt ✅


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/EME_project/models/* models/

### Dipendenze e caricamento del modello

Installo `kornia`, utilizzata nel modulo di early fusion, e importo la classe `Model` di YOLOv5.

In [ ]:
%pip install kornia
from models.yolo import Model

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 15.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 38.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [kornia]


### Configurazione dello student

Istanzio YOLOv5-Small dal file YAML che ho adattato per EME: 6 canali di ingresso (RGB + termico), 6 classi e `earlyFusionConv` come primo modulo. Il file YAML è esterno al notebook.

In [ ]:
model = Model(cfg="models/yolov5s_eme.yaml")


                 from  n    params  module                                  arguments                     
  0                -1  1      4402  models.common.earlyFusionConv           [6, 32, 6, 2, 2]              
  1                -1  1     18560  models.common.Conv                      [32, 64, 3, 2]                
  2                -1  1     18816  models.common.C3                        [64, 64, 1]                   
  3                -1  1     73984  models.common.Conv                      [64, 128, 3, 2]               
  4                -1  2    115712  models.common.C3                        [128, 128, 2]                 
  5                -1  1    295424  models.common.Conv                      [128, 256, 3, 2]              
  6                -1  3    625152  models.common.C3                        [256, 256, 3]                 
  7                -1  1   1180672  models.common.Conv                      [256, 512, 3, 2]              
  8                -1  1   1182720  

### Verifica del primo modulo

Controllo la struttura del layer iniziale e i parametri letti dal file YAML, per essere sicuro che il modello stia usando davvero ShaPE e non il normale primo layer di YOLOv5.

In [ ]:
print(model.model[0]) #shape

earlyFusionConv(
  (conv): Conv2d(3, 32, kernel_size=(6, 6), stride=(2, 2), padding=(2, 2), bias=False)
  (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
  (act): SiLU(inplace=True)
  (sobel): Sobel(normalized=True)
  (max_edge): MaxPool3d(kernel_size=(2, 3, 3), stride=(1, 1, 1), padding=(0, 1, 1), dilation=1, ceil_mode=False)
  (ssim): SSIM()
  (softmax): Softmax(dim=1)
  (conv0): Conv2d(6, 3, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), bias=False)
)


In [ ]:
print("Canali di ingresso:", model.yaml["ch"])
print("Numero di classi:", model.yaml["nc"])
print("Primo modulo:", type(model.model[0]).__name__)

Canali di ingresso: 6
Numero di classi: 6
Primo modulo: earlyFusionConv


### Layer del backbone da utilizzare

Individuo i livelli del backbone che mi interessano per WeakSup. Scelgo i layer 4, 6 e 9 perché forniscono feature a tre scale diverse (1/8, 1/16 e 1/32); controllo anche quanti canali hanno. È un adattamento a YOLOv5 dei livelli usati nell'implementazione originale su ResNet.

In [ ]:
for i in [4, 6, 9, 17, 20, 23, 24]:
    print(i, model.model[i].type)

4 models.common.C3
6 models.common.C3
9 models.common.SPPF
17 models.common.C3
20 models.common.C3
23 models.common.C3
24 models.yolo.Detect


In [ ]:
print("Layer 4:", model.model[4].cv3.conv.out_channels)
print("Layer 6:", model.model[6].cv3.conv.out_channels)
print("Layer 9:", model.model[9].cv2.conv.out_channels)

Layer 4: 128
Layer 6: 256
Layer 9: 512


### Prima prova della Classification Head

La testa di classificazione applica un global average pooling e un layer lineare. Qui uso una feature casuale con 128 canali per verificare soltanto che l'output abbia un valore per ciascuna delle 6 classi.

In [ ]:
from models.eme_weak_sup import EMEClassificationHead
classification_head = EMEClassificationHead(
    in_channels=128,
    num_classes=6
)

import torch

x = torch.randn(2, 128, 80, 80)

output = classification_head(x)

print("Input:", x.shape)
print("Output:", output.shape)

Input: torch.Size([2, 128, 80, 80])
Output: torch.Size([2, 6])


### Prima prova della Segmentation Head

La testa di segmentazione deve invece mantenere l'informazione spaziale: da una feature `[2, 128, 80, 80]` mi aspetto 6 mappe `[80, 80]` per immagine. Per ora sono solo output non addestrati.

In [ ]:
from models.eme_weak_sup import EMESegmentationHead

segmentation_head = EMESegmentationHead(
    in_channels=128,
    num_classes=6
)

x = torch.randn(2, 128, 80, 80)
output = segmentation_head(x)

print(output.shape)

torch.Size([2, 6, 80, 80])


### Recuperare una feature intermedia con un hook

Registro un `forward hook` sul layer 4. Ogni volta che quel layer viene eseguito, la funzione salva il suo output nel dizionario `features`, senza dover modificare il `forward` originale di YOLOv5.

In [ ]:
features = {}

def salva_feature(modulo, input, output):
    features["layer4"] = output

hook = model.model[4].register_forward_hook(salva_feature)

### Test dell'hook

Eseguo un passaggio con un input RGB-T sintetico di `256 × 256` pixel. Il layer 4 produce una feature a risoluzione 1/8, quindi `32 × 32`. Dopo il test rimuovo l'hook: la feature già salvata resta disponibile, ma le esecuzioni successive non la aggiorneranno.

In [ ]:
model.eval()

device = next(model.parameters()).device
x = torch.rand(1, 6, 256, 256, device=device)

with torch.no_grad():
    _ = model(x)

hook.remove()

print("Feature layer 4:", features["layer4"].shape)

Feature layer 4: torch.Size([1, 128, 32, 32])


### Collegamento delle teste alla feature reale

Ora passo alle teste la feature estratta dal layer 4, invece di crearla casualmente. È un primo controllo del collegamento con il backbone: qui uso `eval()` e `no_grad()`, quindi non sto effettuando alcun addestramento.

In [ ]:
feat = features["layer4"]
classification_head = classification_head.to(device).eval()
segmentation_head = segmentation_head.to(device).eval()

with torch.no_grad():
    cls_output = classification_head(feat)
    seg_output = segmentation_head(feat)

print("Classification:", cls_output.shape)
print("Segmentation:", seg_output.shape)

Classification: torch.Size([1, 6])
Segmentation: torch.Size([1, 6, 32, 32])


### WeakSup su tre livelli del backbone

Passo dalla singola feature alle tre scale scelte. Gli hook raccolgono gli output dei layer 4, 6 e 9; il modulo `EMEWeakSupHeads` usa tre convoluzioni 1×1 per adattare i canali e applicare le due teste condivise.

Da qui uso input di `512 × 640` pixel, cioè la risoluzione indicata negli esperimenti di EME. Anche in questo caso non sto ancora caricando il dataset né calcolando le loss.

In [ ]:
import importlib
import models.eme_weak_sup as weak_sup

importlib.reload(weak_sup)

EMEWeakSupHeads = weak_sup.EMEWeakSupHeads

features = {}
handles = []

def make_hook(layer_id):
    def hook(module, inputs, output):
        features[layer_id] = output
    return hook

for layer_id in [4, 6, 9]:
    handle = model.model[layer_id].register_forward_hook(
        make_hook(layer_id)
    )
    handles.append(handle)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device).eval()

weak_heads = EMEWeakSupHeads(
    num_classes=6
).to(device).eval()

x = torch.rand(1, 6, 512, 640, device=device)

with torch.no_grad():
    _ = model(x)

    cls_outputs, seg_outputs = weak_heads([
        features[4],
        features[6],
        features[9]
    ])



### Controllo delle dimensioni in uscita

Verifico che, per ciascun livello, la classificazione restituisca 6 valori e la segmentazione una mappa per classe. Le risoluzioni attese sono `64 × 80`, `32 × 40` e `16 × 20`.

In [ ]:
for i in range(3):
    print(f"Livello {i}:")
    print(" Classification:", cls_outputs[i].shape)
    print(" Segmentation:  ", seg_outputs[i].shape)

Livello 0:
 Classification: torch.Size([1, 6])
 Segmentation:   torch.Size([1, 6, 64, 80])
Livello 1:
 Classification: torch.Size([1, 6])
 Segmentation:   torch.Size([1, 6, 32, 40])
Livello 2:
 Classification: torch.Size([1, 6])
 Segmentation:   torch.Size([1, 6, 16, 20])


### Rimozione degli hook

Finita la verifica, scollego gli hook registrati. Se dovessi ripetere il test in una nuova esecuzione, dovrò registrarli di nuovo.

In [ ]:
for handle in handles:
    handle.remove()